# TSFS12 Hand-in exercise 4: Collaborative Control 6.1

In [43]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
from collab_functions import CreateAgent, AgentFormation

It is necessary to plot in an external windows for the animations to work.
Run the ipython-command below to activate plotting in external windows.

If you have no default Matplotlib backend, you might have to write

```%matplotlib tk``` 

to explicitly state which backend to use (tk is a good choice that should work on all platforms)

In [44]:
%matplotlib qt

# Model, controllers, and agent formation
A first example are 4 agents, all modeled by a single integrator, and all measuring their absolute position.  The controller for each agent is a proportional controller.

In [45]:
def single_integrator(t, x, u, model_params):
    '''
    System-model functions
     
     Input arguments are:
     t - time 
     x - agent state
     u - the control input u, 
     model_params - structure with parameters of the model '''

    return u

def controller_p_v(y, y_ref, ctrl_params):
    # Absolute position controller for the top agent
    Kp = ctrl_params["kp"]
    return Kp * (y_ref - y)


def controller_relative(y, y_ref, ctrl_params):
    Kp = ctrl_params["kp"]
    u = np.zeros(2)

    for relative_position, desired_distance in zip(y, y_ref):
        distance = np.linalg.norm(relative_position)
        u += 4 * Kp * (distance**2 - desired_distance**2) * relative_position

    return u

# Controllers with measurement errors
MEASUREMENT_ERROR = 1.2

# Circle
def controller_relative_agent_4(y, y_ref, ctrl_params):
    y = y.copy()
    y[0] *= MEASUREMENT_ERROR    # Edge (2, 4)
    return controller_relative(y, y_ref, ctrl_params)

# Upwards
def controller_relative_agent_7(y, y_ref, ctrl_params):
    y = y.copy()
    y[:2] *= MEASUREMENT_ERROR  # Edges (5, 7) and (6, 7)
    return controller_relative(y, y_ref, ctrl_params)

# Rotation: agent 1 measures edge (0, 1), and agent 6 measures edge (7, 6).
def controller_relative_agent_1(y, y_ref, ctrl_params):
    y = y.copy()
    y[0] *= MEASUREMENT_ERROR  # Edge (0, 1)
    return controller_relative(y, y_ref, ctrl_params)

def controller_relative_agent_6(y, y_ref, ctrl_params):
    y = y.copy()
    y[2] *= MEASUREMENT_ERROR  # Edge (7, 6)
    return controller_relative(y, y_ref, ctrl_params)


Now, formulate the formation graph and position references. The graph is specified by a set of edges. An edge (i, j) represents that agent j meauseras the distance from agent i. If agent i measures its own absolute position, this is represented by a tuple with element i as (i,) and this is the case for all agents in this example.

In [46]:
n = 2  # (px, py)
G = [ #(i, j) means agent j measures agent i
    (1, 0), (2, 0), # Agent 0 (top) no longer absolute

    # Minimally rigid graph
    (0, 1), (0, 2),
    (1, 2), (1, 3),
    (2, 3), (2, 4),
    (3, 4), (3, 5),
    (4, 5), (4, 6),
    (5, 6), (5, 7),
    (6, 7),

    (2, 1), (3, 1),
    (3, 2), (4, 2),
    (4, 3), (5, 3),
    (5, 4), (6, 4),
    (6, 5), (7, 5),
    (7, 6),
]

desired_positions = np.array([
    [0, 6], [-1, 5], [1, 5], [-1, 3],
    [1, 3], [-1, 1], [1, 1], [0, 0]
])

def distance_reference(agent_idx):
    neighbors = [edge[0] for edge in G if len(edge) == 2 and edge[1] == agent_idx]
    return np.array([
        np.linalg.norm(desired_positions[neighbor] - desired_positions[agent_idx])
        for neighbor in neighbors
    ])

formation_references = [
    lambda t, i=0: distance_reference(i),
    lambda t, i=1: distance_reference(i),
    lambda t, i=2: distance_reference(i),
    lambda t, i=3: distance_reference(i),
    lambda t, i=4: distance_reference(i),
    lambda t, i=5: distance_reference(i),
    lambda t, i=6: distance_reference(i),
    lambda t, i=7: distance_reference(i),
]


Create all agents by assigning model and controller using ```CreateAgent``` and then form the formation using ```AgentFormation``` by providing the references.

# Simulate and animate
Define an initial state and evaluate the formatation state-transition function

In [47]:
# Controller parameters
ctrl_params = {
    "kp": 1.0,
}

# Create agents
agents = []

# Agents use relative measurements
for i in range(8):
    if i == 1:
        controller = controller_relative_agent_1
    elif i == 6:
        controller = controller_relative_agent_6
    else:
        controller = controller_relative
    agents.append(CreateAgent(single_integrator, {}, controller, ctrl_params))

# Create complete formation
formation = AgentFormation(
    agents,
    G,
    formation_references,
    n
)

# Start in the reference shape translated by [2, 0], with its center at (2, 3).
x0 = (desired_positions + np.array([2.0, 0.0])).reshape(-1)

Simulate the formation

In [ ]:
t = np.arange(0, 20, 0.05) # Time vector for the simulation
x = odeint(formation.ode, x0, t)

Animate the results

In [49]:
fig, (ax, ax_dist) = plt.subplots(
    1, 2, num=20, clear=True, figsize=(12, 6),
    gridspec_kw={"width_ratios": [1.2, 1]}
)
fig.subplots_adjust(left=0.06, right=0.98, wspace=0.28)
ax.axis([-2, 6, -1, 7])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Distance-based formation control (Kp = {ctrl_params['kp']})")

# The two biased measurements are on edges (0, 1) and (7, 6).
distance_0_1 = np.linalg.norm(x[:, 0:2] - x[:, 2:4], axis=1)
distance_7_6 = np.linalg.norm(x[:, 14:16] - x[:, 12:14], axis=1)
target_distance_0_1 = np.linalg.norm(desired_positions[0] - desired_positions[1])
ax_dist.plot(t, distance_0_1, color="tab:blue", label="Actual distance (0, 1)")
ax_dist.plot(t, MEASUREMENT_ERROR * distance_0_1, color="tab:blue", linestyle=":", label="Agent 1 measured (0, 1)")
ax_dist.plot(t, distance_7_6, color="tab:green", label="Actual distance (7, 6)")
ax_dist.plot(t, MEASUREMENT_ERROR * distance_7_6, color="tab:green", linestyle=":", label="Agent 6 measured (7, 6)")
ax_dist.axis([0, t[-1], 0, 2.5])
ax_dist.axhline(target_distance_0_1, color="tab:red", linestyle="--", label="Target distance")
ax_dist.set_xlabel("Time [s]")
ax_dist.set_ylabel("Distance")
ax_dist.set_title("Distances on erroneous edges")
ax_dist.grid(True)
ax_dist.legend()

# One moving marker and one trajectory line per agent
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:cyan", "tab:purple", "tab:brown", "tab:pink"]
agents_plot = [ax.plot([], [], "o", color=colors[i])[0] for i in range(len(agents))]
trails = [ax.plot([], [], "-", color=colors[i], alpha=0.7)[0] for i in range(len(agents))]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
def animate(i):
    for idx, (marker, trail) in enumerate(zip(agents_plot, trails)):
        px = x[: i + 1, n * idx]
        py = x[: i + 1, n * idx + 1]
        marker.set_data([px[-1]], [py[-1]])
        trail.set_data(px, py)
    time_text.set_text(f"t = {t[i]:.1f}")
    return tuple(agents_plot + trails) + (time_text,)

ani = animation.FuncAnimation(
    fig, animate, interval=20, frames=x.shape[0], blit=True, repeat=False
)
xf = x[-1].reshape(8, 2)

for i in range(8):
    print(i, xf[i])

0 [2.00896127 9.81742186]
1 [1.00460478 8.84734272]
2 [3.010771   8.84460017]
3 [0.98942963 6.86593562]
4 [2.98882612 6.86344433]
5 [0.95584151 4.89716663]
6 [3.02578739 4.89496084]
7 [1.99020502 4.29242275]
